In [55]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import json
import joblib
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 02 — IDM Selector Comparison

This notebook compares feasible methods for selecting or combining IDM candidate predictions. The goal is to determine whether learned selectors can improve the IDM prior beyond the global IDM and simple regime-aware assignment. The strongest feasible selector is later used as an official IDM variant and as a possible hybrid prior.

In [56]:
# ============================================================
# CONFIG
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"

OUTPUT_DIR = DATA_DIR / "idm_selector_comparison_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# THRESHOLD + HORIZON CONFIG
# ============================================================

BRAKE_THRESHOLD = -0.1
ACCEL_THRESHOLD = 0.2

FPS = 25
HISTORY = 25

# Change this value for each horizon experiment:
# 1   = 0.04 s
# 13  = 0.52 s
# 25  = 1.00 s
# 50  = 2.00 s
# 75  = 3.00 s
# 125 = 5.00 s
HORIZON = 125

STRIDE = 1

THRESHOLD_TAG = (
    f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}"
    .replace(".", "p")
)

HORIZON_SECONDS = HORIZON / FPS
HORIZON_TAG = f"H{HISTORY}_Hz{HORIZON}_S{STRIDE}"

# Same parent folder, but horizon-specific run title/folder
RUN_DIR = OUTPUT_DIR / f"idm_selector_comparison_{THRESHOLD_TAG}_{HORIZON_TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Run dir:", RUN_DIR)
print("Brake threshold:", BRAKE_THRESHOLD)
print("Accel threshold:", ACCEL_THRESHOLD)
print("History frames:", HISTORY)
print("Horizon frames:", HORIZON)
print("Horizon seconds:", HORIZON_SECONDS)
print("Stride:", STRIDE)
print("Horizon tag:", HORIZON_TAG)

assert DATA_DIR.exists(), f"Data directory does not exist: {DATA_DIR}"

Data dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2_H25_Hz125_S1
Brake threshold: -0.1
Accel threshold: 0.2
History frames: 25
Horizon frames: 125
Horizon seconds: 5.0
Stride: 1
Horizon tag: H25_Hz125_S1


In [57]:
# ============================================================
# LOAD PRE-SPLIT DATASETS
# ============================================================

required_files = [
    DATA_DIR / "episode_tracks_train.csv",
    DATA_DIR / "episode_tracks_val.csv",
    DATA_DIR / "episode_tracks_test.csv",
]

for p in required_files:
    assert p.exists(), f"Missing required file: {p}"

episode_tracks_train = pd.read_csv(DATA_DIR / "episode_tracks_train.csv")
episode_tracks_val   = pd.read_csv(DATA_DIR / "episode_tracks_val.csv")
episode_tracks_test  = pd.read_csv(DATA_DIR / "episode_tracks_test.csv")

print("train:", episode_tracks_train.shape)
print("val  :", episode_tracks_val.shape)
print("test :", episode_tracks_test.shape)

train: (11094352, 44)
val  : (2370107, 44)
test : (2368174, 44)


In [58]:
# ============================================================
# FEATURE PREPARATION
# ============================================================

def prepare_base_features(df):
    out = df.copy()

    out["v_ego"] = out["xVelocity"].abs()
    out["v_leader"] = out["precedingXVelocity"].abs()
    out["a_true"] = out["xAcceleration"]

    no_leader = (
        out["precedingId"].isna()
        | (out["precedingId"] <= 0)
        | out["precedingXVelocity"].isna()
        | out["dhw"].isna()
    )

    out["delta_v"] = out["v_ego"] - out["v_leader"]
    out["gap"] = out["dhw"].clip(lower=0.1)

    out.loc[no_leader, "gap"] = 1e6
    out.loc[no_leader, "delta_v"] = 0.0
    out.loc[no_leader, "v_leader"] = out.loc[no_leader, "v_ego"]

    out["no_leader"] = no_leader.astype(int)

    out = out.sort_values(["episode_id", "frame"]).reset_index(drop=True)

    return out


train_base = prepare_base_features(episode_tracks_train)
val_base   = prepare_base_features(episode_tracks_val)
test_base  = prepare_base_features(episode_tracks_test)

print(train_base[[
    "episode_id", "frame", "v_ego", "v_leader",
    "delta_v", "gap", "a_true", "no_leader"
]].head())

   episode_id  frame  v_ego  v_leader  delta_v    gap  a_true  no_leader
0           1  21549  41.19     43.10    -1.91  29.23   -0.39          0
1           1  21550  41.20     43.13    -1.93  29.46   -0.39          0
2           1  21551  41.22     43.15    -1.93  29.64   -0.40          0
3           1  21552  41.23     43.18    -1.95  29.78   -0.41          0
4           1  21553  41.25     43.20    -1.95  29.90   -0.42          0


In [59]:
# ============================================================
# ALIGN IDM DATA TO PREDICTION HORIZON
# IDM input is taken at time t
# Target acceleration is taken at time t + HORIZON
# ============================================================

def make_horizon_aligned_idm_df(df, horizon, stride=1):
    """
    Build horizon-aligned rows for IDM selector evaluation.

    Input features are from time t:
        v_ego, v_leader, delta_v, gap

    Target acceleration is from time t + horizon:
        a_true = future xAcceleration

    The operation is done episode-by-episode to avoid crossing episode boundaries.
    """

    aligned_parts = []

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        if len(g) <= horizon:
            continue

        input_rows = g.iloc[:-horizon:stride].copy().reset_index(drop=True)
        target_rows = g.iloc[horizon::stride].copy().reset_index(drop=True)

        n = min(len(input_rows), len(target_rows))
        input_rows = input_rows.iloc[:n].copy()
        target_rows = target_rows.iloc[:n].copy()

        # Keep current input frame and future target frame
        input_rows["input_frame"] = input_rows["frame"].values
        input_rows["target_frame"] = target_rows["frame"].values

        # Replace current acceleration with future target acceleration
        input_rows["a_true"] = target_rows["a_true"].values

        # Future regime is based on future target acceleration
        input_rows["true_regime"] = np.select(
            [
                input_rows["a_true"] < BRAKE_THRESHOLD,
                input_rows["a_true"] > ACCEL_THRESHOLD,
            ],
            [
                "braking",
                "accelerating",
            ],
            default="steady",
        )

        input_rows["horizon_frames"] = horizon
        input_rows["horizon_seconds"] = horizon / FPS
        input_rows["stride"] = stride

        aligned_parts.append(input_rows)

    if len(aligned_parts) == 0:
        return pd.DataFrame()

    return pd.concat(aligned_parts, ignore_index=True)


train_h = make_horizon_aligned_idm_df(
    train_base,
    horizon=HORIZON,
    stride=STRIDE,
)

val_h = make_horizon_aligned_idm_df(
    val_base,
    horizon=HORIZON,
    stride=STRIDE,
)

test_h = make_horizon_aligned_idm_df(
    test_base,
    horizon=HORIZON,
    stride=STRIDE,
)

print("=== HORIZON-ALIGNED SELECTOR DATA ===")
print("HORIZON:", HORIZON, "frames =", HORIZON_SECONDS, "seconds")
print("STRIDE:", STRIDE)
print("Train rows:", len(train_h))
print("Val rows:", len(val_h))
print("Test rows:", len(test_h))

print("\nTrain future-regime counts:")
print(train_h["true_regime"].value_counts())

print("\nVal future-regime counts:")
print(val_h["true_regime"].value_counts())

print("\nTest future-regime counts:")
print(test_h["true_regime"].value_counts())

print("\nExample aligned rows:")
print(
    train_h[
        [
            "episode_id",
            "input_frame",
            "target_frame",
            "v_ego",
            "v_leader",
            "delta_v",
            "gap",
            "a_true",
            "true_regime",
        ]
    ].head()
)

=== HORIZON-ALIGNED SELECTOR DATA ===
HORIZON: 125 frames = 5.0 seconds
STRIDE: 1
Train rows: 6735602
Val rows: 1436357
Test rows: 1434299

Train future-regime counts:
true_regime
steady          3425715
accelerating    1872925
braking         1436962
Name: count, dtype: int64

Val future-regime counts:
true_regime
steady          725055
accelerating    404584
braking         306718
Name: count, dtype: int64

Test future-regime counts:
true_regime
steady          742802
accelerating    395118
braking         296379
Name: count, dtype: int64

Example aligned rows:
   episode_id  input_frame  target_frame  v_ego  v_leader  delta_v    gap  \
0           1        21549         21674  41.19     43.10    -1.91  29.23   
1           1        21550         21675  41.20     43.13    -1.93  29.46   
2           1        21551         21676  41.22     43.15    -1.93  29.64   
3           1        21552         21677  41.23     43.18    -1.95  29.78   
4           1        21553         21678  41.

In [60]:
# ============================================================
# IDM ACCELERATION
# ============================================================

def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    s0 = float(params["s0"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    delta = float(params["delta"])

    v = np.maximum(v, 0.0)
    s = np.maximum(s, 0.1)

    s_star = s0 + np.maximum(
        0.0,
        v * T + (v * delta_v) / (2.0 * np.sqrt(a * b))
    )

    acc = a * (
        1.0
        - (v / v0) ** delta
        - (s_star / s) ** 2
    )

    acc = np.clip(acc, -8.0, 4.0)

    return acc

In [61]:
# ============================================================
# FINAL IDM PARAMETER SETS
# ============================================================

best_global_params = {"v0": 40.0, "T": 0.5,"s0": 0.5, "a": 0.4,"b": 5.0,"delta": 4.0}

best_params_braking = {'v0': 1.0, 'T': 0.4, 's0': 5.0, 'a': 0.05, 'b': 26.0, 'delta': 0.5}

best_params_accel = {'v0': 110.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}

best_params_steady = {'v0': 44.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 4.0}



In [62]:
# ============================================================
# SELECTOR FEATURE ENGINEERING
# ============================================================

def add_selector_features(df):
    out = df.copy()

    out["time_headway"] = out["gap"] / np.maximum(out["v_ego"], 0.1)
    out["closing_speed"] = np.maximum(out["delta_v"], 0.0)
    out["opening_speed"] = np.maximum(-out["delta_v"], 0.0)
    out["abs_delta_v"] = np.abs(out["delta_v"])
    out["inv_ttc"] = out["closing_speed"] / np.maximum(out["gap"], 0.1)
    out["log_gap"] = np.log1p(np.minimum(out["gap"], 500.0))

    return out


selector_cols = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "time_headway",
    "closing_speed",
    "opening_speed",
    "abs_delta_v",
    "inv_ttc",
    "log_gap"
]

train_sel = add_selector_features(train_h)
val_sel   = add_selector_features(val_h)
test_sel  = add_selector_features(test_h)

print("Selector features:")
print(selector_cols)

print(train_sel[selector_cols + ["a_true"]].head())

Selector features:
['v_ego', 'v_leader', 'delta_v', 'gap', 'time_headway', 'closing_speed', 'opening_speed', 'abs_delta_v', 'inv_ttc', 'log_gap']
   v_ego  v_leader  delta_v    gap  time_headway  closing_speed  \
0  41.19     43.10    -1.91  29.23      0.709638            0.0   
1  41.20     43.13    -1.93  29.46      0.715049            0.0   
2  41.22     43.15    -1.93  29.64      0.719068            0.0   
3  41.23     43.18    -1.95  29.78      0.722290            0.0   
4  41.25     43.20    -1.95  29.90      0.724848            0.0   

   opening_speed  abs_delta_v  inv_ttc   log_gap  a_true  
0           1.91         1.91      0.0  3.408835   -0.14  
1           1.93         1.93      0.0  3.416414   -0.14  
2           1.93         1.93      0.0  3.422306   -0.15  
3           1.95         1.95      0.0  3.426865   -0.15  
4           1.95         1.95      0.0  3.430756   -0.15  


In [63]:
# ============================================================
# TRUE REGIME LABELS
# Future regime based on horizon-aligned target acceleration
# ============================================================

def add_true_regime(df, true_col="a_true"):
    out = df.copy()

    out["true_regime"] = np.where(
        out[true_col] < BRAKE_THRESHOLD,
        "braking",
        np.where(
            out[true_col] > ACCEL_THRESHOLD,
            "accelerating",
            "steady"
        )
    )

    return out


train_sel = add_true_regime(train_sel)
val_sel   = add_true_regime(val_sel)
test_sel  = add_true_regime(test_sel)

print("Train future-regime counts:")
print(train_sel["true_regime"].value_counts())

print("\nValidation future-regime counts:")
print(val_sel["true_regime"].value_counts())

print("\nTest future-regime counts:")
print(test_sel["true_regime"].value_counts())

Train future-regime counts:
true_regime
steady          3425715
accelerating    1872925
braking         1436962
Name: count, dtype: int64

Validation future-regime counts:
true_regime
steady          725055
accelerating    404584
braking         306718
Name: count, dtype: int64

Test future-regime counts:
true_regime
steady          742802
accelerating    395118
braking         296379
Name: count, dtype: int64


In [64]:
# ============================================================
# ADD ALL IDM CANDIDATE PREDICTIONS
# ============================================================

def add_all_idm_candidates(df):
    out = df.copy()

    out["a_idm_global"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_global_params
    )

    out["a_idm_braking"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_braking
    )

    out["a_idm_accel"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_accel
    )

    out["a_idm_steady"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_steady
    )

    return out


train_sel = add_all_idm_candidates(train_sel)
val_sel   = add_all_idm_candidates(val_sel)
test_sel  = add_all_idm_candidates(test_sel)

candidate_cols = [
    "a_idm_global",
    "a_idm_braking",
    "a_idm_accel",
    "a_idm_steady"
]

print(test_sel[candidate_cols].describe())

       a_idm_global  a_idm_braking   a_idm_accel  a_idm_steady
count  1.434299e+06   1.434299e+06  1.434299e+06  1.434299e+06
mean   1.620414e-01  -2.217881e-01  3.854553e-01  4.710924e-02
std    3.363020e-01   7.428783e-02  4.159300e-02  1.150519e-01
min   -8.000000e+00  -2.603108e+00 -1.295453e+00 -4.493973e+00
25%    1.201641e-01  -2.464639e-01  3.880491e-01  5.042188e-02
50%    2.571648e-01  -2.162481e-01  3.981465e-01  7.783726e-02
75%    3.351035e-01  -1.917310e-01  3.999968e-01  9.080811e-02
max    3.999499e-01   4.151850e-02  4.000000e-01  9.999900e-02


In [65]:
# ============================================================
# IDM METRIC HELPERS
# ============================================================

def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    corr = np.nan
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Bias": float(np.mean(y_pred - y_true)),
        "Corr": corr
    }


def evaluate_idm_prediction(df, pred_col="a_idm_selected", model_name="model"):
    overall = compute_regression_metrics(df["a_true"], df[pred_col])
    overall["Model"] = model_name

    regime_rows = []

    for regime, g in df.groupby("true_regime"):
        m = compute_regression_metrics(g["a_true"], g[pred_col])
        m["Model"] = model_name
        m["Regime"] = regime
        m["Count"] = len(g)
        regime_rows.append(m)

    overall_df = pd.DataFrame([overall])
    regime_df = pd.DataFrame(regime_rows)

    regime_order = ["accelerating", "braking", "steady"]
    regime_df["Regime"] = pd.Categorical(
        regime_df["Regime"],
        categories=regime_order,
        ordered=True
    )

    regime_df = regime_df.sort_values("Regime").reset_index(drop=True)

    return overall_df, regime_df

In [66]:
# ============================================================
# SELECTED IDM PRIOR FROM LABEL
# ============================================================

def selected_idm_from_label(df, label_col):
    out = df.copy()

    out["a_idm_selected"] = np.nan

    out.loc[out[label_col] == "global", "a_idm_selected"] = out.loc[
        out[label_col] == "global", "a_idm_global"
    ]

    out.loc[out[label_col] == "braking", "a_idm_selected"] = out.loc[
        out[label_col] == "braking", "a_idm_braking"
    ]

    out.loc[out[label_col] == "accelerating", "a_idm_selected"] = out.loc[
        out[label_col] == "accelerating", "a_idm_accel"
    ]

    out.loc[out[label_col] == "steady", "a_idm_selected"] = out.loc[
        out[label_col] == "steady", "a_idm_steady"
    ]

    if out["a_idm_selected"].isna().any():
        missing = out.loc[out["a_idm_selected"].isna(), label_col].unique()
        raise ValueError(f"Some labels could not be mapped: {missing}")

    return out

In [67]:
# ============================================================
# BASELINE IDM REFERENCES
# ============================================================

baseline_overall_rows = []
baseline_regime_rows = []

# 1. Global IDM
global_tmp = test_sel.copy()
global_tmp["a_idm_selected"] = global_tmp["a_idm_global"]

overall_df, regime_df = evaluate_idm_prediction(
    global_tmp,
    pred_col="a_idm_selected",
    model_name="Global_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

# 2. True-regime oracle IDM
oracle_tmp = test_sel.copy()
oracle_tmp["selector_label"] = oracle_tmp["true_regime"]
oracle_tmp = selected_idm_from_label(oracle_tmp, "selector_label")

overall_df, regime_df = evaluate_idm_prediction(
    oracle_tmp,
    pred_col="a_idm_selected",
    model_name="TrueRegime_Oracle_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

# 3. Ideal best-candidate IDM
def add_best_idm_label(df):
    out = df.copy()

    candidates = {
        "global": "a_idm_global",
        "braking": "a_idm_braking",
        "accelerating": "a_idm_accel",
        "steady": "a_idm_steady"
    }

    errors = np.column_stack([
        np.abs(out["a_true"].values - out[col].values)
        for col in candidates.values()
    ])

    best_idx = np.argmin(errors, axis=1)
    label_names = np.array(list(candidates.keys()))

    out["best_idm_label"] = label_names[best_idx]
    out["best_idm_error"] = errors[np.arange(len(out)), best_idx]

    return out


train_best = add_best_idm_label(train_sel)
val_best   = add_best_idm_label(val_sel)
test_best  = add_best_idm_label(test_sel)

ideal_best_tmp = test_best.copy()
ideal_best_tmp["selector_label"] = ideal_best_tmp["best_idm_label"]
ideal_best_tmp = selected_idm_from_label(ideal_best_tmp, "selector_label")

overall_df, regime_df = evaluate_idm_prediction(
    ideal_best_tmp,
    pred_col="a_idm_selected",
    model_name="Ideal_BestCandidate_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

baseline_overall_df = pd.concat(baseline_overall_rows, ignore_index=True)
baseline_regime_df = pd.concat(baseline_regime_rows, ignore_index=True)

print("=== BASELINE IDM REFERENCES ===")
print(
    baseline_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

print("\nBest-IDM label counts on test:")
print(test_best["best_idm_label"].value_counts())

=== BASELINE IDM REFERENCES ===
                  Model      MAE     RMSE    MedAE        R2     Bias      Corr
Ideal_BestCandidate_IDM 0.112789 0.216742 0.066349  0.590979 0.007112  0.787744
  TrueRegime_Oracle_IDM 0.129983 0.230967 0.084447  0.535527 0.014599  0.738431
             Global_IDM 0.336028 0.496769 0.252741 -1.148666 0.084273 -0.051444

Best-IDM label counts on test:
best_idm_label
steady          594811
braking         320416
accelerating    283447
global          235625
Name: count, dtype: int64


In [68]:
# ============================================================
# REGIME SELECTOR MODELS
# ============================================================

selector_models = {
    "DecisionTree_regime": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=500,
        random_state=SEED,
        class_weight="balanced"
    ),

    "RandomForest_regime": RandomForestClassifier(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=300,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced_subsample"
    ),

    "ExtraTrees_regime": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=200,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced"
    ),

    "HistGB_regime": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=SEED
    ),

    "KNN_regime": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=51,
            weights="distance",
            n_jobs=-1
        ))
    ])
}

In [69]:
# ============================================================
# TRAIN REGIME SELECTORS AND EVALUATE DOWNSTREAM IDM
# ============================================================

MAX_TRAIN_ROWS_FOR_KNN = 250_000

regime_selector_overall_rows = []
regime_selector_regime_rows = []
regime_classification_rows = []

X_train_full = train_sel[selector_cols]
y_train_regime_full = train_sel["true_regime"]

X_test = test_sel[selector_cols]
y_test_regime = test_sel["true_regime"]

for name, clf in selector_models.items():
    print(f"\n==============================")
    print(f"Training selector: {name}")
    print(f"==============================")

    if name.startswith("KNN"):
        sample_idx = train_sel.sample(
            n=min(MAX_TRAIN_ROWS_FOR_KNN, len(train_sel)),
            random_state=SEED
        ).index

        X_train_use = train_sel.loc[sample_idx, selector_cols]
        y_train_use = train_sel.loc[sample_idx, "true_regime"]
    else:
        X_train_use = X_train_full
        y_train_use = y_train_regime_full

    clf.fit(X_train_use, y_train_use)

    pred_test = clf.predict(X_test)

    print("\nClassification report on test:")
    print(classification_report(y_test_regime, pred_test, digits=4))

    tmp = test_sel.copy()
    tmp["selector_label"] = pred_test
    tmp = selected_idm_from_label(tmp, "selector_label")

    overall_df, regime_df = evaluate_idm_prediction(
        tmp,
        pred_col="a_idm_selected",
        model_name=name
    )

    regime_selector_overall_rows.append(overall_df)
    regime_selector_regime_rows.append(regime_df)

    regime_classification_rows.append({
        "Model": name,
        "Accuracy": np.mean(pred_test == y_test_regime),
        "Pred_accelerating": np.sum(pred_test == "accelerating"),
        "Pred_braking": np.sum(pred_test == "braking"),
        "Pred_steady": np.sum(pred_test == "steady")
    })


regime_selector_overall_df = pd.concat(regime_selector_overall_rows, ignore_index=True)
regime_selector_regime_df = pd.concat(regime_selector_regime_rows, ignore_index=True)
regime_classification_df = pd.DataFrame(regime_classification_rows)

print("\n=== REGIME SELECTOR DOWNSTREAM IDM OVERALL RESULTS ===")
print(
    regime_selector_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


Training selector: DecisionTree_regime

Classification report on test:
              precision    recall  f1-score   support

accelerating     0.3726    0.5291    0.4373    395118
     braking     0.3548    0.4973    0.4141    296379
      steady     0.7440    0.4584    0.5673    742802

    accuracy                         0.4859   1434299
   macro avg     0.4905    0.4950    0.4729   1434299
weighted avg     0.5613    0.4859    0.4998   1434299


Training selector: RandomForest_regime

Classification report on test:
              precision    recall  f1-score   support

accelerating     0.3872    0.4985    0.4359    395118
     braking     0.3581    0.4767    0.4090    296379
      steady     0.7217    0.5160    0.6017    742802

    accuracy                         0.5030   1434299
   macro avg     0.4890    0.4971    0.4822   1434299
weighted avg     0.5544    0.5030    0.5162   1434299


Training selector: ExtraTrees_regime

Classification report on test:
              precision 

In [70]:
# ============================================================
# BEST-IDM SELECTOR MODELS
# ============================================================

best_selector_models = {
    "DecisionTree_bestIDM": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=500,
        random_state=SEED,
        class_weight="balanced"
    ),

    "RandomForest_bestIDM": RandomForestClassifier(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=300,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced_subsample"
    ),

    "ExtraTrees_bestIDM": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=200,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced"
    ),

    "HistGB_bestIDM": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=SEED
    )
}

In [71]:
# ============================================================
# TRAIN BEST-IDM SELECTORS AND EVALUATE DOWNSTREAM IDM
# ============================================================

best_selector_overall_rows = []
best_selector_regime_rows = []
best_classification_rows = []

X_train_best = train_best[selector_cols]
y_train_best = train_best["best_idm_label"]

X_test_best = test_best[selector_cols]
y_test_best = test_best["best_idm_label"]

for name, clf in best_selector_models.items():
    print(f"\n==============================")
    print(f"Training selector: {name}")
    print(f"==============================")

    clf.fit(X_train_best, y_train_best)

    pred_test = clf.predict(X_test_best)

    print("\nBest-IDM label classification report on test:")
    print(classification_report(y_test_best, pred_test, digits=4))

    tmp = test_best.copy()
    tmp["selector_label"] = pred_test
    tmp = selected_idm_from_label(tmp, "selector_label")

    overall_df, regime_df = evaluate_idm_prediction(
        tmp,
        pred_col="a_idm_selected",
        model_name=name
    )

    best_selector_overall_rows.append(overall_df)
    best_selector_regime_rows.append(regime_df)

    best_classification_rows.append({
        "Model": name,
        "Accuracy": np.mean(pred_test == y_test_best),
        "Pred_accelerating": np.sum(pred_test == "accelerating"),
        "Pred_braking": np.sum(pred_test == "braking"),
        "Pred_global": np.sum(pred_test == "global"),
        "Pred_steady": np.sum(pred_test == "steady")
    })


best_selector_overall_df = pd.concat(best_selector_overall_rows, ignore_index=True)
best_selector_regime_df = pd.concat(best_selector_regime_rows, ignore_index=True)
best_classification_df = pd.DataFrame(best_classification_rows)

print("\n=== BEST-IDM SELECTOR DOWNSTREAM IDM OVERALL RESULTS ===")
print(
    best_selector_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


Training selector: DecisionTree_bestIDM

Best-IDM label classification report on test:
              precision    recall  f1-score   support

accelerating     0.4210    0.5028    0.4583    283447
     braking     0.4846    0.4060    0.4418    320416
      global     0.2632    0.5106    0.3474    235625
      steady     0.7212    0.4491    0.5535    594811

    accuracy                         0.4602   1434299
   macro avg     0.4725    0.4671    0.4502   1434299
weighted avg     0.5338    0.4602    0.4759   1434299


Training selector: RandomForest_bestIDM

Best-IDM label classification report on test:
              precision    recall  f1-score   support

accelerating     0.4150    0.5297    0.4654    283447
     braking     0.4588    0.4294    0.4436    320416
      global     0.2725    0.4456    0.3382    235625
      steady     0.7130    0.4645    0.5625    594811

    accuracy                         0.4664   1434299
   macro avg     0.4649    0.4673    0.4524   1434299
weighted 

In [72]:
# ============================================================
# COMBINE OVERALL RESULTS
# ============================================================

all_overall_df = pd.concat(
    [
        baseline_overall_df,
        regime_selector_overall_df,
        best_selector_overall_df
    ],
    ignore_index=True
)

all_regime_df = pd.concat(
    [
        baseline_regime_df,
        regime_selector_regime_df,
        best_selector_regime_df
    ],
    ignore_index=True
)

print("\n=== ALL IDM SELECTOR OVERALL RESULTS ===")
print(
    all_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


=== ALL IDM SELECTOR OVERALL RESULTS ===
                  Model      MAE     RMSE    MedAE        R2     Bias      Corr
Ideal_BestCandidate_IDM 0.112789 0.216742 0.066349  0.590979 0.007112  0.787744
  TrueRegime_Oracle_IDM 0.129983 0.230967 0.084447  0.535527 0.014599  0.738431
          HistGB_regime 0.230560 0.350783 0.147515 -0.071364 0.016472  0.143480
         HistGB_bestIDM 0.236254 0.357454 0.152287 -0.112497 0.024922  0.125750
             KNN_regime 0.243251 0.364816 0.158237 -0.158797 0.018123  0.106399
     ExtraTrees_bestIDM 0.247854 0.366999 0.166235 -0.172707 0.053612  0.128754
   DecisionTree_bestIDM 0.248260 0.366679 0.167920 -0.170663 0.061329  0.131002
   RandomForest_bestIDM 0.251003 0.369506 0.170395 -0.188784 0.052733  0.125538
      ExtraTrees_regime 0.269679 0.389596 0.190000 -0.321567 0.034338  0.128570
    RandomForest_regime 0.274010 0.393263 0.195938 -0.346563 0.029745  0.118334
    DecisionTree_regime 0.282211 0.400704 0.209434 -0.398001 0.037110  0.11404

In [73]:
# ============================================================
# XGBOOST BEST-IDM SELECTOR: HARD AND SOFT MIXTURE
# ============================================================

le_best = LabelEncoder()

y_train_xgb = le_best.fit_transform(train_best["best_idm_label"])
y_test_xgb  = le_best.transform(test_best["best_idm_label"])

xgb_best = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_best.fit(train_best[selector_cols], y_train_xgb)

# -------------------------
# Hard selection
# -------------------------
pred_idx = xgb_best.predict(test_best[selector_cols])
pred_label = le_best.inverse_transform(pred_idx.astype(int))

xgb_hard_tmp = test_best.copy()
xgb_hard_tmp["selector_label"] = pred_label
xgb_hard_tmp = selected_idm_from_label(xgb_hard_tmp, "selector_label")

xgb_hard_overall, xgb_hard_regime = evaluate_idm_prediction(
    xgb_hard_tmp,
    pred_col="a_idm_selected",
    model_name="XGB_bestIDM_hard"
)

# -------------------------
# Soft mixture
# -------------------------
proba = xgb_best.predict_proba(test_best[selector_cols])

xgb_soft_tmp = test_best.copy()
xgb_soft_tmp["a_idm_soft"] = 0.0

# Store XGB probabilities and hard label for later analysis/export
for j, cls in enumerate(le_best.classes_):
    xgb_soft_tmp[f"p_{cls}"] = proba[:, j]

hard_idx = np.argmax(proba, axis=1)
xgb_soft_tmp["xgb_best_idm_hard_label"] = le_best.inverse_transform(hard_idx)

class_to_col = {
    "global": "a_idm_global",
    "braking": "a_idm_braking",
    "accelerating": "a_idm_accel",
    "steady": "a_idm_steady"
}

for j, cls in enumerate(le_best.classes_):
    xgb_soft_tmp["a_idm_soft"] += proba[:, j] * xgb_soft_tmp[class_to_col[cls]].values

xgb_soft_overall, xgb_soft_regime = evaluate_idm_prediction(
    xgb_soft_tmp,
    pred_col="a_idm_soft",
    model_name="XGB_bestIDM_soft"
)

print("\n=== XGBOOST BEST-IDM HARD VS SOFT ===")
print(
    pd.concat([xgb_hard_overall, xgb_soft_overall], ignore_index=True)[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)

print("\n=== XGBOOST BEST-IDM BY REGIME ===")
print(
    pd.concat([xgb_hard_regime, xgb_soft_regime], ignore_index=True)[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)


=== XGBOOST BEST-IDM HARD VS SOFT ===
           Model      MAE     RMSE    MedAE        R2     Bias     Corr
XGB_bestIDM_hard 0.236025 0.357236 0.152026 -0.111141 0.023834 0.125573
XGB_bestIDM_soft 0.216863 0.333856 0.138491  0.029537 0.005665 0.173600

=== XGBOOST BEST-IDM BY REGIME ===
           Model       Regime      MAE     RMSE      Bias  Count
XGB_bestIDM_hard accelerating 0.323242 0.437112 -0.300204 395118
XGB_bestIDM_soft accelerating 0.339066 0.437400 -0.339041 395118
XGB_bestIDM_soft      braking 0.420010 0.514202  0.418863 296379
XGB_bestIDM_hard      braking 0.426036 0.548918  0.422738 296379
XGB_bestIDM_soft       steady 0.070804 0.089197  0.024157 742802
XGB_bestIDM_hard       steady 0.113818 0.156724  0.037037 742802


In [74]:
# ============================================================
# UPDATE ALL RESULTS WITH XGBOOST
# ============================================================

all_overall_with_xgb_df = pd.concat(
    [
        all_overall_df,
        xgb_hard_overall,
        xgb_soft_overall
    ],
    ignore_index=True
)

all_regime_with_xgb_df = pd.concat(
    [
        all_regime_df,
        xgb_hard_regime,
        xgb_soft_regime
    ],
    ignore_index=True
)

print("\n=== ALL IDM SELECTOR OVERALL RESULTS INCLUDING XGBOOST ===")
print(
    all_overall_with_xgb_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

print("\n=== TOP 5 IDM SELECTORS PER REGIME INCLUDING XGBOOST ===")
best_per_regime_xgb = (
    all_regime_with_xgb_df
    .sort_values("MAE")
    .groupby("Regime")
    .head(5)
    .reset_index(drop=True)
)

print(
    best_per_regime_xgb[
        ["Regime", "Model", "MAE", "RMSE", "Bias", "Count"]
    ].to_string(index=False)
)


=== ALL IDM SELECTOR OVERALL RESULTS INCLUDING XGBOOST ===
                  Model      MAE     RMSE    MedAE        R2     Bias      Corr
Ideal_BestCandidate_IDM 0.112789 0.216742 0.066349  0.590979 0.007112  0.787744
  TrueRegime_Oracle_IDM 0.129983 0.230967 0.084447  0.535527 0.014599  0.738431
       XGB_bestIDM_soft 0.216863 0.333856 0.138491  0.029537 0.005665  0.173600
          HistGB_regime 0.230560 0.350783 0.147515 -0.071364 0.016472  0.143480
       XGB_bestIDM_hard 0.236025 0.357236 0.152026 -0.111141 0.023834  0.125573
         HistGB_bestIDM 0.236254 0.357454 0.152287 -0.112497 0.024922  0.125750
             KNN_regime 0.243251 0.364816 0.158237 -0.158797 0.018123  0.106399
     ExtraTrees_bestIDM 0.247854 0.366999 0.166235 -0.172707 0.053612  0.128754
   DecisionTree_bestIDM 0.248260 0.366679 0.167920 -0.170663 0.061329  0.131002
   RandomForest_bestIDM 0.251003 0.369506 0.170395 -0.188784 0.052733  0.125538
      ExtraTrees_regime 0.269679 0.389596 0.190000 -0.321567

In [75]:
# ============================================================
# XGBOOST TRUE-REGIME SELECTOR: HARD AND SOFT MIXTURE
# ============================================================

le_regime = LabelEncoder()

y_train_xgb_regime = le_regime.fit_transform(train_sel["true_regime"])
y_test_xgb_regime  = le_regime.transform(test_sel["true_regime"])

xgb_regime = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_regime.fit(train_sel[selector_cols], y_train_xgb_regime)

# -------------------------
# Hard regime selection
# -------------------------
pred_idx = xgb_regime.predict(test_sel[selector_cols])
pred_regime = le_regime.inverse_transform(pred_idx.astype(int))

xgb_regime_hard_tmp = test_sel.copy()
xgb_regime_hard_tmp["selector_label"] = pred_regime
xgb_regime_hard_tmp = selected_idm_from_label(
    xgb_regime_hard_tmp,
    "selector_label"
)

xgb_regime_hard_overall, xgb_regime_hard_regime = evaluate_idm_prediction(
    xgb_regime_hard_tmp,
    pred_col="a_idm_selected",
    model_name="XGB_regime_hard"
)

print("\n=== XGBOOST REGIME CLASSIFICATION REPORT ===")
print(classification_report(
    test_sel["true_regime"],
    pred_regime,
    digits=4
))

# -------------------------
# Soft regime mixture
# -------------------------
proba = xgb_regime.predict_proba(test_sel[selector_cols])

xgb_regime_soft_tmp = test_sel.copy()
xgb_regime_soft_tmp["a_idm_soft"] = 0.0

regime_to_col = {
    "braking": "a_idm_braking",
    "accelerating": "a_idm_accel",
    "steady": "a_idm_steady"
}

for j, cls in enumerate(le_regime.classes_):
    xgb_regime_soft_tmp["a_idm_soft"] += (
        proba[:, j] * xgb_regime_soft_tmp[regime_to_col[cls]].values
    )

xgb_regime_soft_overall, xgb_regime_soft_regime = evaluate_idm_prediction(
    xgb_regime_soft_tmp,
    pred_col="a_idm_soft",
    model_name="XGB_regime_soft"
)

print("\n=== XGBOOST REGIME HARD VS SOFT IDM ===")
print(
    pd.concat(
        [xgb_regime_hard_overall, xgb_regime_soft_overall],
        ignore_index=True
    )[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)

print("\n=== XGBOOST REGIME HARD VS SOFT BY REGIME ===")
print(
    pd.concat(
        [xgb_regime_hard_regime, xgb_regime_soft_regime],
        ignore_index=True
    )[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)


=== XGBOOST REGIME CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

accelerating     0.4501    0.2807    0.3458    395118
     braking     0.5124    0.2615    0.3463    296379
      steady     0.6002    0.8376    0.6993    742802

    accuracy                         0.5651   1434299
   macro avg     0.5209    0.4599    0.4638   1434299
weighted avg     0.5407    0.5651    0.5289   1434299


=== XGBOOST REGIME HARD VS SOFT IDM ===
          Model      MAE     RMSE    MedAE        R2     Bias     Corr
XGB_regime_hard 0.230418 0.350806 0.147404 -0.071505 0.014492 0.143590
XGB_regime_soft 0.221078 0.338993 0.141146 -0.000558 0.013662 0.109065

=== XGBOOST REGIME HARD VS SOFT BY REGIME ===
          Model       Regime      MAE     RMSE      Bias  Count
XGB_regime_hard accelerating 0.329189 0.437373 -0.309673 395118
XGB_regime_soft accelerating 0.335929 0.441354 -0.335868 395118
XGB_regime_hard      braking 0.420973 0.536409  0.415933 296379
XGB_regime_soft 

In [76]:
# ============================================================
# HISTGB TRUE-REGIME SELECTOR: SOFT MIXTURE
# ============================================================

histgb_regime_soft_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.03,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=SEED
)

histgb_regime_soft_model.fit(
    train_sel[selector_cols],
    train_sel["true_regime"]
)

# Hard
histgb_regime_pred = histgb_regime_soft_model.predict(test_sel[selector_cols])

histgb_regime_hard_tmp = test_sel.copy()
histgb_regime_hard_tmp["selector_label"] = histgb_regime_pred
histgb_regime_hard_tmp = selected_idm_from_label(
    histgb_regime_hard_tmp,
    "selector_label"
)

histgb_regime_hard_overall, histgb_regime_hard_regime = evaluate_idm_prediction(
    histgb_regime_hard_tmp,
    pred_col="a_idm_selected",
    model_name="HistGB_regime_hard_tuned"
)

# Soft
histgb_proba = histgb_regime_soft_model.predict_proba(test_sel[selector_cols])

histgb_regime_soft_tmp = test_sel.copy()
histgb_regime_soft_tmp["a_idm_soft"] = 0.0

for j, cls in enumerate(histgb_regime_soft_model.classes_):
    histgb_regime_soft_tmp["a_idm_soft"] += (
        histgb_proba[:, j] * histgb_regime_soft_tmp[regime_to_col[cls]].values
    )

histgb_regime_soft_overall, histgb_regime_soft_regime = evaluate_idm_prediction(
    histgb_regime_soft_tmp,
    pred_col="a_idm_soft",
    model_name="HistGB_regime_soft"
)

print("\n=== HISTGB REGIME HARD VS SOFT IDM ===")
print(
    pd.concat(
        [histgb_regime_hard_overall, histgb_regime_soft_overall],
        ignore_index=True
    )[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)


=== HISTGB REGIME HARD VS SOFT IDM ===
                   Model      MAE     RMSE    MedAE        R2     Bias     Corr
HistGB_regime_hard_tuned 0.230377 0.350226 0.147647 -0.067963 0.016876 0.143651
      HistGB_regime_soft 0.221013 0.338808 0.141236  0.000535 0.013822 0.110611


In [77]:
# ============================================================
# FINAL EXTENDED COMBINATION OF ALL RESULTS
# ============================================================

all_overall_extended_df = pd.concat(
    [
        all_overall_with_xgb_df,
        xgb_regime_hard_overall,
        xgb_regime_soft_overall,
        histgb_regime_hard_overall,
        histgb_regime_soft_overall
    ],
    ignore_index=True
)

all_regime_extended_df = pd.concat(
    [
        all_regime_with_xgb_df,
        xgb_regime_hard_regime,
        xgb_regime_soft_regime,
        histgb_regime_hard_regime,
        histgb_regime_soft_regime
    ],
    ignore_index=True
)

print("\n=== FINAL EXTENDED IDM SELECTOR OVERALL RESULTS ===")
print(
    all_overall_extended_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


=== FINAL EXTENDED IDM SELECTOR OVERALL RESULTS ===
                   Model      MAE     RMSE    MedAE        R2     Bias      Corr
 Ideal_BestCandidate_IDM 0.112789 0.216742 0.066349  0.590979 0.007112  0.787744
   TrueRegime_Oracle_IDM 0.129983 0.230967 0.084447  0.535527 0.014599  0.738431
        XGB_bestIDM_soft 0.216863 0.333856 0.138491  0.029537 0.005665  0.173600
      HistGB_regime_soft 0.221013 0.338808 0.141236  0.000535 0.013822  0.110611
         XGB_regime_soft 0.221078 0.338993 0.141146 -0.000558 0.013662  0.109065
HistGB_regime_hard_tuned 0.230377 0.350226 0.147647 -0.067963 0.016876  0.143651
         XGB_regime_hard 0.230418 0.350806 0.147404 -0.071505 0.014492  0.143590
           HistGB_regime 0.230560 0.350783 0.147515 -0.071364 0.016472  0.143480
        XGB_bestIDM_hard 0.236025 0.357236 0.152026 -0.111141 0.023834  0.125573
          HistGB_bestIDM 0.236254 0.357454 0.152287 -0.112497 0.024922  0.125750
              KNN_regime 0.243251 0.364816 0.158237 -0.1

In [78]:
# ============================================================
# BEST MODEL PER REGIME - FINAL EXTENDED TABLE
# ============================================================

print("\n=== FINAL EXTENDED IDM SELECTOR RESULTS BY REGIME ===")
print(
    all_regime_extended_df[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)

best_per_regime_extended = (
    all_regime_extended_df
    .sort_values("MAE")
    .groupby("Regime")
    .head(5)
    .reset_index(drop=True)
)

print("\n=== FINAL TOP 5 IDM SELECTORS PER REGIME BY MAE ===")
print(
    best_per_regime_extended[
        ["Regime", "Model", "MAE", "RMSE", "Bias", "Count"]
    ].to_string(index=False)
)


=== FINAL EXTENDED IDM SELECTOR RESULTS BY REGIME ===
                   Model       Regime      MAE     RMSE      Bias  Count
 Ideal_BestCandidate_IDM accelerating 0.147288 0.276208 -0.095998 395118
   TrueRegime_Oracle_IDM accelerating 0.171590 0.284770 -0.052200 395118
    DecisionTree_bestIDM accelerating 0.282730 0.411204 -0.256021 395118
      ExtraTrees_bestIDM accelerating 0.292309 0.421264 -0.264702 395118
    RandomForest_bestIDM accelerating 0.294414 0.423951 -0.265094 395118
          HistGB_bestIDM accelerating 0.322395 0.436587 -0.299102 395118
        XGB_bestIDM_hard accelerating 0.323242 0.437112 -0.300204 395118
HistGB_regime_hard_tuned accelerating 0.326374 0.433770 -0.306808 395118
           HistGB_regime accelerating 0.327149 0.434964 -0.307173 395118
         XGB_regime_hard accelerating 0.329189 0.437373 -0.309673 395118
       ExtraTrees_regime accelerating 0.334386 0.474828 -0.282016 395118
      HistGB_regime_soft accelerating 0.335644 0.441022 -0.335573 395

In [79]:
# ============================================================
# FINAL CONSISTENCY CHECK
# ============================================================

print("BRAKE_THRESHOLD:", BRAKE_THRESHOLD)
print("ACCEL_THRESHOLD:", ACCEL_THRESHOLD)
print("RUN_DIR:", RUN_DIR)

print("\nFinal overall table shape:", all_overall_extended_df.shape)
print("Final regime table shape :", all_regime_extended_df.shape)

print("\nModels included in final overall table:")
for model_name in sorted(all_overall_extended_df["Model"].unique()):
    print("-", model_name)

print("\nFinal overall results sorted by MAE:")
print(
    all_overall_extended_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

BRAKE_THRESHOLD: -0.1
ACCEL_THRESHOLD: 0.2
RUN_DIR: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2_H25_Hz125_S1

Final overall table shape: (18, 7)
Final regime table shape : (54, 9)

Models included in final overall table:
- DecisionTree_bestIDM
- DecisionTree_regime
- ExtraTrees_bestIDM
- ExtraTrees_regime
- Global_IDM
- HistGB_bestIDM
- HistGB_regime
- HistGB_regime_hard_tuned
- HistGB_regime_soft
- Ideal_BestCandidate_IDM
- KNN_regime
- RandomForest_bestIDM
- RandomForest_regime
- TrueRegime_Oracle_IDM
- XGB_bestIDM_hard
- XGB_bestIDM_soft
- XGB_regime_hard
- XGB_regime_soft

Final overall results sorted by MAE:
                   Model      MAE     RMSE    MedAE        R2     Bias      Corr
 Ideal_BestCandidate_IDM 0.112789 0.216742 0.066349  0.590979 0.007112  0.787744
   TrueRegime_Oracle_IDM 0.129983 0.230967 0.084447  0.535527 0.014599  0.738431
        XGB_bestIDM_soft 0.2168

In [80]:
# ============================================================
# SAVE FINAL EXTENDED RESULTS
# Horizon-specific outputs
# ============================================================

all_overall_extended_df.to_csv(
    RUN_DIR / f"idm_selector_overall_results_{HORIZON_TAG}.csv",
    index=False
)

all_regime_extended_df.to_csv(
    RUN_DIR / f"idm_selector_regime_results_{HORIZON_TAG}.csv",
    index=False
)

regime_classification_df.to_csv(
    RUN_DIR / f"regime_selector_classification_summary_{HORIZON_TAG}.csv",
    index=False
)

best_classification_df.to_csv(
    RUN_DIR / f"best_idm_selector_classification_summary_{HORIZON_TAG}.csv",
    index=False
)

with open(RUN_DIR / f"config_{HORIZON_TAG}.json", "w") as f:
    json.dump({
        "BRAKE_THRESHOLD": BRAKE_THRESHOLD,
        "ACCEL_THRESHOLD": ACCEL_THRESHOLD,
        "FPS": FPS,
        "HISTORY": HISTORY,
        "HORIZON": HORIZON,
        "HORIZON_SECONDS": HORIZON_SECONDS,
        "STRIDE": STRIDE,
        "THRESHOLD_TAG": THRESHOLD_TAG,
        "HORIZON_TAG": HORIZON_TAG,
        "DATA_DIR": str(DATA_DIR),
        "RUN_DIR": str(RUN_DIR),
        "selector_cols": selector_cols,
        "candidate_cols": candidate_cols,
        "best_global_params": best_global_params,
        "best_params_braking": best_params_braking,
        "best_params_accel": best_params_accel,
        "best_params_steady": best_params_steady,
        "xgb_best_classes": list(le_best.classes_)
    }, f, indent=2)

print("Saved final extended results to:")
print(RUN_DIR)
print("Horizon tag:", HORIZON_TAG)

Saved final extended results to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2_H25_Hz125_S1
Horizon tag: H25_Hz125_S1


In [81]:
# ============================================================
# SAVE XGB-SOFT BEST-IDM TEST PREDICTIONS
# Horizon-specific output
# ============================================================

xgb_soft_predictions = xgb_soft_tmp.copy()

xgb_soft_predictions = xgb_soft_predictions.rename(columns={
    "a_true": "y_true",
    "a_idm_soft": "y_pred"
})

xgb_soft_predictions["Model"] = "XGB-soft best-IDM selector"
xgb_soft_predictions["error"] = (
    xgb_soft_predictions["y_pred"] - xgb_soft_predictions["y_true"]
)

xgb_soft_pred_path = RUN_DIR / f"xgb_soft_best_idm_test_predictions_{HORIZON_TAG}.csv"

xgb_soft_predictions.to_csv(
    xgb_soft_pred_path,
    index=False
)

# Save selector bundle too, so later notebooks can load the correct horizon version
xgb_soft_bundle_path = RUN_DIR / f"xgb_soft_best_idm_selector_bundle_{HORIZON_TAG}.joblib"

joblib.dump(
    {
        "model": xgb_best,
        "label_encoder": le_best,
        "selector_cols": selector_cols,
        "classes": list(le_best.classes_),
        "idm_param_tag": "refinedIDM_v2",
        "threshold_tag": THRESHOLD_TAG,
        "horizon_tag": HORIZON_TAG,
        "history": HISTORY,
        "horizon": HORIZON,
        "stride": STRIDE,
        "fps": FPS,
    },
    xgb_soft_bundle_path
)

print("Saved XGB-soft IDM test predictions to:")
print(xgb_soft_pred_path)

print("Saved XGB-soft selector bundle to:")
print(xgb_soft_bundle_path)

Saved XGB-soft IDM test predictions to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2_H25_Hz125_S1\xgb_soft_best_idm_test_predictions_H25_Hz125_S1.csv
Saved XGB-soft selector bundle to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2_H25_Hz125_S1\xgb_soft_best_idm_selector_bundle_H25_Hz125_S1.joblib
